# Feature Engineering for Power BI Dashboard
## Minimal Essential Features Only

**Goal:** Create only the features needed for the 3-page Power BI dashboard.

**Features Created (only 3):**
1. `customer_region` - Geographic region (Southeast, South, etc.)
2. `customer_segment` - New/Regular/Loyal customer
3. `is_repeat_customer` - True/False flag

**Input:** cleaned_olist_data.csv (32 columns)  
**Output:** dashboard_ready.csv (35 columns)

---
## Step 1: Import Libraries and Load Data

In [1]:
import pandas as pd
import os

# Define paths
PROCESSED_PATH = r"C:\Users\KIIT\Desktop\projects\olist-analytics\data\processed"

# Load cleaned data
df = pd.read_csv(os.path.join(PROCESSED_PATH, 'cleaned_olist_data.csv'))

print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")

Loaded: 112,650 rows x 32 columns


---
## Step 2: Create Customer Region

Maps customer_state to Brazilian regions for geographic analysis.

In [2]:
# Brazilian state to region mapping
brazil_regions = {
    'SP': 'Southeast', 'RJ': 'Southeast', 'MG': 'Southeast', 'ES': 'Southeast',
    'PR': 'South', 'SC': 'South', 'RS': 'South',
    'BA': 'Northeast', 'PE': 'Northeast', 'CE': 'Northeast',
    'RN': 'Northeast', 'PB': 'Northeast', 'AL': 'Northeast',
    'SE': 'Northeast', 'PI': 'Northeast', 'MA': 'Northeast',
    'AM': 'North', 'PA': 'North', 'AC': 'North', 'RO': 'North',
    'RR': 'North', 'AP': 'North', 'TO': 'North',
    'GO': 'Central-West', 'DF': 'Central-West',
    'MT': 'Central-West', 'MS': 'Central-West'
}

# Create customer_region column
df['customer_region'] = df['customer_state'].map(brazil_regions).fillna('Other')

print("Created: customer_region")
print("\nDistribution:")
print(df['customer_region'].value_counts())

Created: customer_region

Distribution:
customer_region
Southeast       77413
South           16151
Northeast       10409
Central-West     6613
North            2064
Name: count, dtype: int64


---
## Step 3: Create Customer Segment

Categorizes customers based on their order count:
- **New:** 1 order
- **Regular:** 2-3 orders
- **Loyal:** 4+ orders

In [3]:
# Use customer_id_unique as the true customer identifier
customer_id_col = 'customer_id_unique' if 'customer_id_unique' in df.columns else 'customer_id'
print(f"Using '{customer_id_col}' as customer identifier")

# Calculate order count per customer
customer_orders = df.groupby(customer_id_col)['order_id'].nunique().reset_index()
customer_orders.columns = [customer_id_col, 'order_count']

# Merge back to main dataframe
df = df.merge(customer_orders, on=customer_id_col, how='left')

# Create customer_segment
def segment_customer(count):
    if count == 1:
        return 'New'
    elif count <= 3:
        return 'Regular'
    else:
        return 'Loyal'

df['customer_segment'] = df['order_count'].apply(segment_customer)

print("Created: customer_segment")
print("\nDistribution:")
print(df['customer_segment'].value_counts())

Using 'customer_id_unique' as customer identifier
Created: customer_segment

Distribution:
customer_segment
New        105167
Regular      7174
Loyal         309
Name: count, dtype: int64


---
## Step 4: Create is_repeat_customer Flag

Simple True/False flag to identify repeat customers.

In [4]:
# Create is_repeat_customer flag
df['is_repeat_customer'] = df['order_count'] > 1

print("Created: is_repeat_customer")

repeat_count = df['is_repeat_customer'].sum()
total_customers = df[customer_id_col].nunique()
print(f"\nRepeat customers: {repeat_count:,} out of {total_customers:,}")
print(f"Repeat customer rate: {repeat_count/total_customers*100:.1f}%")

# Drop the temporary order_count column
df = df.drop(columns=['order_count'])

Created: is_repeat_customer

Repeat customers: 7,483 out of 95,420
Repeat customer rate: 7.8%


---
## Step 5: Save Dashboard-Ready Dataset

In [6]:
# Save final dataset
output_file = os.path.join(PROCESSED_PATH, 'olist_data.csv')
df.to_csv(output_file, index=False)

print("=" * 60)
print("DASHBOARD-READY DATASET SAVED")
print("=" * 60)
print(f"\nFile: {output_file}")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print(f"\nNew columns added: 3")
print("  1. customer_region")
print("  2. customer_segment")
print("  3. is_repeat_customer")

DASHBOARD-READY DATASET SAVED

File: C:\Users\KIIT\Desktop\projects\olist-analytics\data\processed\olist_data.csv
Rows: 112,650
Columns: 35

New columns added: 3
  1. customer_region
  2. customer_segment
  3. is_repeat_customer


In [7]:
# Verify the saved file
df_verify = pd.read_csv(output_file, nrows=5)
print(f"\nVerification: File loaded successfully ({len(df_verify)} rows, {len(df_verify.columns)} columns)")


Verification: File loaded successfully (5 rows, 35 columns)


---
## Summary

**Features Created:**
| Feature | Purpose | 
|---------|---------|
| customer_region | Geographic grouping |
| customer_segment | Customer loyalty | 
| is_repeat_customer | Repeat flag |

